Constraints and CAD go hand in hand. There are two high level types of constraints: ones we apply to sketches and ones we apply to assemblies. Sketch constraints are things like "make this line parallel to that line" or "make this circle tangent to that circle". Assembly constraints are things like "make this part revolute to that part" or "make this part linear to that part".

In [ ]:
from codetocad.adapters.build123d import Part, Assembly
import build123d as bd

# Create parts
base = Part.preset.cube(10, 10, 2)
base.set_name("base")

cylinder = Part.preset.cylinder(2, 5)
cylinder.set_name("cylinder")

# Create assembly
assembly = Assembly("example_assembly")
assembly.add_part(base)
assembly.add_part(cylinder)

# Create rigid mate using fluent API
location1 = bd.Location((0, 0, 1))  # Location on base
location2 = bd.Location((0, 0, 0))  # Location on cylinder

rigid_mate = assembly.mate.rigid(
    base, cylinder, location1=location1, location2=location2, name="fix_cylinder"
)

print(f"Created mate: {rigid_mate.name}")
print(f"Status: {rigid_mate.status.value}")

In [ ]:
from codetocad.adapters.blender.cli import run_blender


def test_blender_mates_comprehensive():
    """
    Comprehensive test of Blender assembly mates with real Blender objects.
    This function will be executed inside Blender using run_blender.
    """
    import bpy
    import bmesh
    from mathutils import Vector, Euler
    import sys
    import os

    # Add the project root to Python path
    project_root = os.path.dirname(os.path.dirname(os.path.abspath(__file__)))
    if project_root not in sys.path:
        sys.path.insert(0, project_root)

    print("=" * 60)
    print("BLENDER ASSEMBLY MATES COMPREHENSIVE TEST")
    print("=" * 60)

    try:
        # Clear existing scene
        bpy.ops.object.select_all(action="SELECT")
        bpy.ops.object.delete(use_global=False)

        # Import CodeToCAD Blender adapter
        from codetocad.adapters.blender import Assembly, Part

        print("✓ Successfully imported CodeToCAD Blender adapter")

        # Test 1: Create Assembly and Parts with Real Geometry
        print("\n1. Creating Assembly and Parts with Real Geometry")
        print("-" * 50)

        assembly = Assembly("test_robotic_arm")
        print(f"✓ Created assembly: {assembly.name}")

        # Create parts with actual Blender geometry
        base_part = Part("base")
        shoulder_part = Part("shoulder")
        upper_arm_part = Part("upper_arm")
        forearm_part = Part("forearm")
        wrist_part = Part("wrist")

        # Create real geometry for each part
        parts_geometry = [
            (base_part, "base", (0, 0, 0), (2, 2, 0.5)),
            (shoulder_part, "shoulder", (0, 0, 1), (1, 1, 1)),
            (upper_arm_part, "upper_arm", (0, 0, 3), (0.5, 0.5, 2)),
            (forearm_part, "forearm", (0, 0, 6), (0.4, 0.4, 1.5)),
            (wrist_part, "wrist", (0, 0, 8), (0.3, 0.3, 0.5)),
        ]

        created_objects = []
        for part, name, location, dimensions in parts_geometry:
            # Create cube mesh
            bpy.ops.mesh.primitive_cube_add(location=location)
            obj = bpy.context.active_object
            obj.name = name
            obj.scale = dimensions

            # Apply scale
            bpy.context.view_layer.objects.active = obj
            bpy.ops.object.transform_apply(location=False, rotation=False, scale=True)

            # Set the part's Blender object
            part._blender_object = obj
            # Constraints will be initialized automatically when accessed

            created_objects.append(obj)
            print(f"✓ Created part: {name} at {location} with dimensions {dimensions}")

        # Add parts to assembly
        for part, _, _, _ in parts_geometry:
            assembly.add(part)

        print(f"✓ Added {len(parts_geometry)} parts to assembly")

        # Test 2: Kinematic Mates
        print("\n2. Testing Kinematic Mates")
        print("-" * 50)

        # Test Rigid Mate
        print("Testing Rigid Mate...")
        try:
            rigid_mate = assembly.mate.rigid(
                part1=base_part,
                part2=shoulder_part,
                location1=(0, 0, 0.5),  # Use tuple instead of Vector
                location2=(0, 0, 0),
                name="base_shoulder_rigid",
            )

            if rigid_mate:
                print(f"✓ Created rigid mate: {rigid_mate.name}")
                print(f"  Degrees of freedom: {rigid_mate.get_degrees_of_freedom()}")

                # Verify constraint was created
                shoulder_obj = shoulder_part.get_blender_object()
                if shoulder_obj and len(shoulder_obj.constraints) > 0:
                    print(
                        f"  ✓ Blender constraints created: {len(shoulder_obj.constraints)}"
                    )
                    for constraint in shoulder_obj.constraints:
                        print(f"    - {constraint.name} ({constraint.type})")
                else:
                    print("  ⚠ No Blender constraints found")
            else:
                print("  ✗ Failed to create rigid mate")

        except Exception as e:
            print(f"  ✗ Rigid mate error: {e}")

        # Test Revolute Mate
        print("\nTesting Revolute Mate...")
        try:
            revolute_mate = assembly.mate.revolute(
                part1=shoulder_part,
                part2=upper_arm_part,
                axis=(0, 1, 0),  # Y-axis rotation
                location1=(0, 0, 0.5),
                location2=(0, 0, -1),
                angle_range=(-90, 90),
                current_angle=0,
                name="shoulder_revolute",
            )

            if revolute_mate:
                print(f"✓ Created revolute mate: {revolute_mate.name}")
                print(f"  Angle range: {revolute_mate.angle_range}")
                print(f"  Current angle: {revolute_mate.current_angle}")
                print(f"  Degrees of freedom: {revolute_mate.get_degrees_of_freedom()}")

                # Test angle control
                test_angle = 45.0
                if revolute_mate.set_angle(test_angle):
                    actual_angle = revolute_mate.get_angle()
                    print(f"  ✓ Set angle to {test_angle}°, got {actual_angle:.1f}°")
                else:
                    print(f"  ⚠ Failed to set angle to {test_angle}°")

                # Verify constraints
                upper_arm_obj = upper_arm_part.get_blender_object()
                if upper_arm_obj and len(upper_arm_obj.constraints) > 0:
                    print(
                        f"  ✓ Blender constraints created: {len(upper_arm_obj.constraints)}"
                    )
                    for constraint in upper_arm_obj.constraints:
                        print(f"    - {constraint.name} ({constraint.type})")
            else:
                print("  ✗ Failed to create revolute mate")

        except Exception as e:
            print(f"  ✗ Revolute mate error: {e}")

        # Test Linear Mate
        print("\nTesting Linear Mate...")
        try:
            linear_mate = assembly.mate.linear(
                part1=upper_arm_part,
                part2=forearm_part,
                axis=(0, 0, 1),  # Z-axis translation
                location1=(0, 0, 1),
                location2=(0, 0, -0.75),
                position_range=(0, 2),
                current_position=1,
                name="elbow_linear",
            )

            if linear_mate:
                print(f"✓ Created linear mate: {linear_mate.name}")
                print(f"  Position range: {linear_mate.position_range}")
                print(f"  Current position: {linear_mate.current_position}")
                print(f"  Degrees of freedom: {linear_mate.get_degrees_of_freedom()}")

                # Test position control
                test_position = 1.5
                if linear_mate.set_position(test_position):
                    actual_position = linear_mate.get_position()
                    print(
                        f"  ✓ Set position to {test_position}, got {actual_position:.1f}"
                    )
                else:
                    print(f"  ⚠ Failed to set position to {test_position}")

                # Verify constraints
                forearm_obj = forearm_part.get_blender_object()
                if forearm_obj and len(forearm_obj.constraints) > 0:
                    print(
                        f"  ✓ Blender constraints created: {len(forearm_obj.constraints)}"
                    )
            else:
                print("  ✗ Failed to create linear mate")

        except Exception as e:
            print(f"  ✗ Linear mate error: {e}")

        # Test Ball Mate
        print("\nTesting Ball Mate...")
        try:
            ball_mate = assembly.mate.ball(
                part1=forearm_part,
                part2=wrist_part,
                center_point=(0, 0, 0.75),
                location1=(0, 0, 0.75),
                location2=(0, 0, -0.25),
                angle_ranges=((-45, 45), (-45, 45), (-90, 90)),
                current_angles=(0, 0, 0),
                name="wrist_ball",
            )

            if ball_mate:
                print(f"✓ Created ball mate: {ball_mate.name}")
                print(f"  Angle ranges: {ball_mate.angle_ranges}")
                print(f"  Current angles: {ball_mate.current_angles}")
                print(f"  Degrees of freedom: {ball_mate.get_degrees_of_freedom()}")

                # Test angle control
                test_angles = (15, -10, 30)
                if ball_mate.set_angles(*test_angles):
                    actual_angles = ball_mate.get_angles()
                    print(f"  ✓ Set angles to {test_angles}, got {actual_angles}")
                else:
                    print(f"  ⚠ Failed to set angles to {test_angles}")

                # Verify constraints
                wrist_obj = wrist_part.get_blender_object()
                if wrist_obj and len(wrist_obj.constraints) > 0:
                    print(
                        f"  ✓ Blender constraints created: {len(wrist_obj.constraints)}"
                    )
            else:
                print("  ✗ Failed to create ball mate")

        except Exception as e:
            print(f"  ✗ Ball mate error: {e}")

        # Test 3: Geometric Mates
        print("\n3. Testing Geometric Mates")
        print("-" * 50)

        # Create additional parts for geometric testing
        part_a = Part("part_a")
        part_b = Part("part_b")

        # Create geometry for geometric mate testing
        bpy.ops.mesh.primitive_cube_add(location=(3, 0, 1))
        obj_a = bpy.context.active_object
        obj_a.name = "part_a"
        part_a._blender_object = obj_a
        # Constraints will be initialized automatically when accessed

        bpy.ops.mesh.primitive_cube_add(location=(5, 0, 1))
        obj_b = bpy.context.active_object
        obj_b.name = "part_b"
        part_b._blender_object = obj_b
        # Constraints will be initialized automatically when accessed

        assembly.add(part_a)
        assembly.add(part_b)

        print("✓ Created additional parts for geometric mate testing")

        # Test Distance Mate
        print("\nTesting Distance Mate...")
        try:
            distance_mate = assembly.mate.distance(
                part1=part_a,
                part2=part_b,
                entity1="face_x_pos",  # Simplified entity reference
                entity2="face_x_neg",
                distance=3.0,
                name="spacing_constraint",
            )

            if distance_mate:
                print(f"✓ Created distance mate: {distance_mate.name}")
                print(f"  Distance: {distance_mate.get_distance()}")

                # Test distance control
                new_distance = 4.0
                if distance_mate.set_distance(new_distance):
                    actual_distance = distance_mate.get_distance()
                    print(f"  ✓ Set distance to {new_distance}, got {actual_distance}")
                else:
                    print(f"  ⚠ Failed to set distance to {new_distance}")

                # Verify constraints
                if obj_b and len(obj_b.constraints) > 0:
                    print(f"  ✓ Blender constraints created for distance mate")
            else:
                print("  ✗ Failed to create distance mate")

        except Exception as e:
            print(f"  ✗ Distance mate error: {e}")

        # Test Parallel Mate
        print("\nTesting Parallel Mate...")
        try:
            parallel_mate = assembly.mate.parallel(
                part1=part_a,
                part2=part_b,
                entity1="face_top",
                entity2="face_top",
                name="parallel_surfaces",
            )

            if parallel_mate:
                print(f"✓ Created parallel mate: {parallel_mate.name}")

                # Verify constraints
                if obj_b and any(c.type == "COPY_ROTATION" for c in obj_b.constraints):
                    print(f"  ✓ Copy rotation constraint created for parallel mate")
            else:
                print("  ✗ Failed to create parallel mate")

        except Exception as e:
            print(f"  ✗ Parallel mate error: {e}")

        # Test 4: Mate Management
        print("\n4. Testing Mate Management")
        print("-" * 50)

        try:
            # Get the mate manager from the assembly (should exist if mates were created)
            if hasattr(assembly, "_mate_manager"):
                mate_manager = assembly._mate_manager
            else:
                # Create a new one if none exists
                from codetocad.adapters.blender.cad.assembly.mate.blender_mate_manager import (
                    BlenderMateManager,
                )

                mate_manager = BlenderMateManager(assembly)
                assembly._mate_manager = mate_manager

            # Get all mates
            all_mates = mate_manager.get_all_mates()
            print(f"✓ Total mates in assembly: {len(all_mates)}")

            for mate in all_mates:
                mate_name = mate.name if hasattr(mate, "name") else "unknown"
                mate_type = (
                    mate.mate_type.value if hasattr(mate, "mate_type") else "unknown"
                )
                print(f"  - {mate_name} ({mate_type})")

            # Validate mates
            validation_results = mate_manager.validate_mates()
            valid_count = sum(1 for is_valid in validation_results.values() if is_valid)
            print(f"✓ Valid mates: {valid_count}/{len(validation_results)}")

            for mate_name, is_valid in validation_results.items():
                status = "Valid" if is_valid else "Invalid"
                print(f"  - {mate_name}: {status}")

            # Solve mates (update dependency graph)
            solve_success = mate_manager.solve_mates()
            print(f"✓ Mate solving: {'successful' if solve_success else 'failed'}")

            # Test mate removal
            if len(all_mates) > 0:
                mate_to_remove = all_mates[0].name
                removal_success = mate_manager.remove_mate(mate_to_remove)
                print(
                    f"✓ Mate removal test: {'successful' if removal_success else 'failed'}"
                )

                # Verify removal
                updated_mates = mate_manager.get_all_mates()
                print(f"  Mates after removal: {len(updated_mates)}")

        except Exception as e:
            print(f"✗ Mate management error: {e}")

        # Test 5: Animation and Real-time Updates
        print("\n5. Testing Animation and Real-time Updates")
        print("-" * 50)

        try:
            # Test frame updates
            original_frame = bpy.context.scene.frame_current

            # Set keyframes for revolute mate if it exists
            if "revolute_mate" in locals() and revolute_mate:
                print("Testing animation with revolute mate...")

                # Set initial keyframe
                bpy.context.scene.frame_set(1)
                revolute_mate.set_angle(0)
                upper_arm_obj = upper_arm_part.get_blender_object()
                if upper_arm_obj:
                    upper_arm_obj.keyframe_insert(data_path="rotation_euler", index=2)

                # Set end keyframe
                bpy.context.scene.frame_set(50)
                revolute_mate.set_angle(90)
                if upper_arm_obj:
                    upper_arm_obj.keyframe_insert(data_path="rotation_euler", index=2)

                print("✓ Created animation keyframes for revolute mate")

                # Test frame updates
                for frame in [1, 25, 50]:
                    bpy.context.scene.frame_set(frame)
                    current_angle = revolute_mate.get_angle()
                    print(f"  Frame {frame}: angle = {current_angle:.1f}°")

            # Restore original frame
            bpy.context.scene.frame_set(original_frame)

        except Exception as e:
            print(f"✗ Animation test error: {e}")

        # Test 6: Constraint Verification
        print("\n6. Verifying Blender Constraints")
        print("-" * 50)

        total_constraints = 0
        constraint_types = {}

        for obj in bpy.data.objects:
            if obj.constraints:
                obj_constraints = len(obj.constraints)
                total_constraints += obj_constraints
                print(f"✓ Object '{obj.name}': {obj_constraints} constraints")

                for constraint in obj.constraints:
                    constraint_type = constraint.type
                    constraint_types[constraint_type] = (
                        constraint_types.get(constraint_type, 0) + 1
                    )
                    print(f"  - {constraint.name} ({constraint_type})")

        print(f"\n✓ Total Blender constraints created: {total_constraints}")
        print("✓ Constraint types used:")
        for ctype, count in constraint_types.items():
            print(f"  - {ctype}: {count}")

        # Final Summary
        print("\n" + "=" * 60)
        print("TEST SUMMARY")
        print("=" * 60)

        print(f"✓ Assembly created with {len(assembly.parts)} parts")
        print(f"✓ Total mates created: {len(mate_manager.get_all_mates())}")
        print(f"✓ Total Blender constraints: {total_constraints}")
        print(f"✓ Constraint types: {len(constraint_types)}")

        # Save the test scene
        test_file_path = "/tmp/blender_mates_test.blend"
        bpy.ops.wm.save_as_mainfile(filepath=test_file_path)
        print(f"✓ Test scene saved to: {test_file_path}")

        print("\n🎉 ALL BLENDER ASSEMBLY MATE TESTS COMPLETED SUCCESSFULLY!")
        return True

    except Exception as e:
        print(f"\n✗ Test failed with error: {e}")
        import traceback

        traceback.print_exc()
        return False


run_blender(test_blender_mates_comprehensive, background=False)